Processing of data in ATSBR. Note the any modification of the file without the use of this notebook requires cell [3] to be rerun for this notebook to properly.

Open "Average_Transit_Speeds_by_Route_Time_of_Day.csv", make can clone, and save it as "atsbr.csv"

In [7]:
import pandas as pd

# Read the original CSV file
df = pd.read_csv("Average_Transit_Speeds_by_Route_Time_of_Day.csv")

# Save a clone as "atsbr.csv"
df.to_csv("atsbr.csv", index=False)

Open "atsbr.csv" with Pandas

In [8]:
import pandas as pd

df = pd.read_csv("atsbr.csv")

Drop all unneeded fields: {"route_id", "base64_url", "org_id", "agency", "route_name"}

In [9]:
df = df.drop(columns=["route_id", "base64_url", "org_id", "agency", "route_name"])
df.to_csv("atsbr.csv", index=False)

Reformat the "district_name" so that it only contains the number eg. "NN" instead of "NN - XXXXXX"

In [11]:
df["district_name"] = df["district_name"].str[:2]
df.to_csv("atsbr.csv", index=False)

Sort the data by "district_name" then "OBJECTID" then "direction_id" then "time_period"


In [13]:
df = df.sort_values(by=["district_name", "OBJECTID", "direction_id", "time_period"])
df.to_csv("atsbr.csv", index=False)

Remove the data if the "district_name" is empty

In [14]:
df = df[df["district_name"].str.strip() != ""]
df.to_csv("atsbr.csv", index=False)

Sort the columns by "OBJECTID" then "direction_id" then "time_period" then "speed_mph" then "Shape_Length" then "district_name"

In [15]:
df = df[["OBJECTID", "direction_id", "time_period", "speed_mph", "Shape_Length", "district_name"]]
df.to_csv("atsbr.csv", index=False)

Remove the data if the "time_period" is "all_day"

In [16]:
df = df[df["time_period"] != "all_day"]
df.to_csv("atsbr.csv", index=False)

Remove standalone data. Meaning that if the OBJECTID of the row above is not the (OBJECTID of the current - 1) and the OBJECTID of the row below is not the (OBJECTID of the current + 1)

In [17]:
# Identify rows where the OBJECTID of the previous row is (current OBJECTID - 1)
prev_is_consecutive = df["OBJECTID"].shift(1) == (df["OBJECTID"] - 1)
# Identify rows where the OBJECTID of the next row is (current OBJECTID + 1)
next_is_consecutive = df["OBJECTID"].shift(-1) == (df["OBJECTID"] + 1)
# Keep rows that are not standalone
df = df[prev_is_consecutive | next_is_consecutive]
df.to_csv("atsbr.csv", index=False)

Drop the "OBJECTID" and "time_period" column

In [20]:
df = df.drop(columns=["OBJECTID", "time_period"])
df.to_csv("atsbr.csv", index=False)

KeyError: "['OBJECTID', 'time_period'] not found in axis"

Merge the data to calculate congestion rate by formulae: [(speed_mph_offpeak) - (speed_mph_peak) / (speed_mph_offpeak)];[(total Shape_Length) = (Shape_Length top) + (Shape_Length bottom)]

In [22]:
# Create a new DataFrame by grouping every two consecutive rows
merged_rows = []

for i in range(0, len(df) - 1, 2):
    top = df.iloc[i]
    bottom = df.iloc[i + 1]
    merged = {
        "direction_id": top["direction_id"],
        "district_name": top["district_name"],
        "Shape_Length": top["Shape_Length"] + bottom["Shape_Length"],
        "speed_mph": (top["speed_mph"] - bottom["speed_mph"]) / top["speed_mph"] if top["speed_mph"] != 0 else None
    }
    merged_rows.append(merged)

merged_df = pd.DataFrame(merged_rows)
merged_df.to_csv("atsbr.csv", index=False)

Drop column {"direction_id"}

In [23]:
merged_df = merged_df.drop(columns=["direction_id"])
merged_df.to_csv("atsbr.csv", index=False)

Change field name of "speed_mph" to "speed_cr", speed congestion rate.

In [24]:
merged_df = merged_df.rename(columns={"speed_mph": "speed_cr"})
merged_df.to_csv("atsbr.csv", index=False)

Merge the rows to find aggregate congestion rate for each "district_name" based on the formula: (District Congestion Index)=[∑(Shape_Length(i))*​(Segment Congestion Index(i))/∑(Shape_Length(i))]

In [25]:
# Group by 'district_name' and calculate the weighted average congestion index
district_agg = merged_df.groupby("district_name").apply(
    lambda g: pd.Series({
        "district_Shape_Length": g["Shape_Length"].sum(),
        "district_congestion_index": (g["Shape_Length"] * g["speed_cr"]).sum() / g["Shape_Length"].sum()
    })
).reset_index()

district_agg.to_csv("atsbr.csv", index=False)

/var/folders/0g/g1btf_jx2p1f1dx9sx5n8c380000gp/T/ipykernel_26348/3349651247.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  district_agg = merged_df.groupby("district_name").apply(


Drop the "district_Shape_Length" column

In [26]:
district_agg = district_agg.drop(columns=["district_Shape_Length"])
district_agg.to_csv("atsbr.csv", index=False)

Change field name of "district_congestion_index" to "speed_acr", aggregate congestion rate.

In [27]:
district_agg = district_agg.rename(columns={"district_congestion_index": "speed_acr"})
district_agg.to_csv("atsbr.csv", index=False)